In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize, least_squares
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from tqdm import tqdm

sns.set_style('ticks', rc={"axes.facecolor": (0, 0, 0, 0)})
sns.set_context('talk')

from matplotlib import rcParams
rcParams['font.family'] = 'DejaVu Sans'
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']

### Check estimates

In [2]:
gf = pd.read_csv("../../out/Growth_features_estimates_failed_samples.tsv", sep="\t")
ge = pd.read_csv("../../out/Growth_features_estimates.tsv", sep="\t")

In [3]:
gf

,ID


In [4]:
ge

,ID,Method,L,r,t0,L0,H,lag
0,D39_1.1,BFGS,0.723305,2.007703,6.481780,0.009661,0.732966,5.021021
1,D39_1.2,BFGS,0.740443,1.619135,5.952469,0.005449,0.745892,4.140140
2,D39_dHCS_1.1,L-BFGS-B,0.669817,1.527635,4.770855,0.004007,0.673824,2.852853
3,D39_dHCS_1.2,BFGS,0.665998,1.420550,4.662000,0.010114,0.676112,2.596096
4,D39_16Pro_1.1,BFGS,0.650248,1.527089,5.917013,0.006855,0.657104,3.996997
...,...,...,...,...,...,...,...,...
655,ATCC17619_4Pro_3.2,BFGS,0.877604,1.321084,5.166526,0.031027,0.908631,2.945946
656,ATCC17619_4IgG_3.1,BFGS,0.697328,1.255046,5.515581,0.055633,0.752961,3.171171
657,ATCC17619_4IgG_3.2,BFGS,0.769627,1.202776,5.725213,0.052641,0.822268,3.280781
658,ATCC17619_4IgG_dHCS_3.1,BFGS,0.705371,1.141188,5.216039,0.087261,0.792632,2.642643


In [5]:
ge['Sample'] = ge['ID'].str.split('.').str[0]
mean_per_sample = ge.groupby('Sample')[['L', 'r', 't0', 'L0', 'H', 'lag']].mean().reset_index()

In [6]:
mean_per_sample

,Sample,L,r,t0,L0,H,lag
0,110,1.023737,1.345280,8.565064,0.018176,1.041912,6.301688
1,1186,0.843779,1.428708,5.974268,-0.003566,0.840212,3.837762
2,1188,0.772158,1.346153,6.201123,-0.007960,0.764197,3.954940
3,1189,0.706749,1.164113,8.265183,-0.000352,0.706397,5.547051
4,1193,0.919552,1.512761,5.865161,0.000661,0.920213,3.761512
...,...,...,...,...,...,...,...
100,R6_8Pro_2,0.532589,1.411525,7.329595,0.000988,0.533577,5.247748
101,R6_8Pro_3,0.453667,1.404037,7.584582,-0.001423,0.452245,5.494494
102,R6_dHCS_1,0.582410,1.354927,7.608628,0.005838,0.588249,5.441441
103,R6_dHCS_2,0.613134,1.245177,6.567397,0.004836,0.617970,4.080080


### Plot estimates per sample

In [7]:
gd_l = pd.read_csv("../../data/data_transformed.tsv", sep="\t")

In [ ]:
# load fitted parameters
params = pd.read_csv("../../out/Growth_features_estimates.tsv", sep="\t", header=0,
    dtype={"ID": str, "Method": str, "L": float, "r": float, "t0": float, "L0": float, "delta.H": float, "lag": float})

# --- logistic function ---
def lgf(L, r, t0, L0, t):
    t = np.asarray(t)
    return L0 + L / (1 + np.exp(-r*(t - t0)))

# --- mask for fitting only ---
def compute_mask(hours, od):
    max_idx = np.nanargmax(od)
    Max_time = hours[max_idx]
    Min_time = 0.0

    mask_main = (od <= od.max()) & (hours <= Max_time) & (hours >= Min_time)
    mask_tail = (np.abs(od - od.max()) / od.max() <= 0.05) & (hours > Max_time)
    keep_mask = mask_main | mask_tail

    if keep_mask.sum() < 4:
        keep_mask = np.ones_like(od, dtype=bool)
    return keep_mask

# --- loop through all samples ---
samples = [s for s in gd_l["Sample"].unique() if s != "Blanc"]

for sample_id in samples:
    sel_sample_all = gd_l[gd_l["Sample"] == sample_id].copy()
    reps = sel_sample_all["ID_FINAL"].unique()

    fig, axes = plt.subplots(1, 4, figsize=(14, 5))
    axes = axes.flatten()

    palette = sns.color_palette("inferno", len(reps))
    color_single = palette[0]  # first replicate for single panel

    # --- panel 1: single replicate raw ---
    rep_to_plot = reps[0]
    sel_rep_single = sel_sample_all[sel_sample_all["ID_FINAL"] == rep_to_plot].sort_values("Hours")
    ax = axes[0]
    ax.scatter(sel_rep_single["Hours"], sel_rep_single["OD"], color=color_single, s=40, alpha=0.8)
    ax.set_title(f"Single replicate\n{rep_to_plot}", size=16)
    ax.set_xlabel("Time (hours)", size=16)
    ax.set_ylabel(r"OD$_{620}$", size=16)
    ax.set_xlim(0, 22)
    ax.set_ylim(0, 1.5)

    # --- panel 2: all replicates raw ---
    ax = axes[1]
    for idx, rep in enumerate(reps):
        sel_rep = sel_sample_all[sel_sample_all["ID_FINAL"] == rep].sort_values("Hours")
        ax.scatter(sel_rep["Hours"], sel_rep["OD"], color=palette[idx], s=30, alpha=0.7, label=rep)
    ax.set_title(f"All replicates\n{sample_id}", size=16)
    ax.set_xlabel("Time (hours)", size=16)
    ax.set_xlim(0, 22)
    ax.set_ylim(0, 1.5)

    # --- panel 3: fitted curve single replicate ---
    row_single = params[params["ID"] == rep_to_plot].iloc[0]
    L_s, r_s, t0_s, L0_s = row_single["L"], row_single["r"], row_single["t0"], row_single["L0"]
    keep_mask = compute_mask(sel_rep_single["Hours"].values, sel_rep_single["OD"].values)
    fit_hours = sel_rep_single["Hours"].values[keep_mask]
    t_dense = np.linspace(fit_hours.min(), fit_hours.max(), 300)
    y_dense = lgf(L_s, r_s, t0_s, L0_s, t_dense)

    ax = axes[2]
    ax.scatter(sel_rep_single["Hours"], sel_rep_single["OD"], color=color_single, s=30, alpha=0.6)
    ax.plot(t_dense, y_dense, color=color_single, linewidth=2.5)
    ax.set_title(f"Fitted curve\n{rep_to_plot}", size=16)
    ax.set_xlabel("Time (hours)", size=16)
    ax.set_xlim(0, 22)
    ax.set_ylim(0, 1.5)

    # --- panel 4: fitted curves all replicates ---
    ax = axes[3]
    for idx, rep in enumerate(reps):
        sel_rep = sel_sample_all[sel_sample_all["ID_FINAL"] == rep].sort_values("Hours")
        hours = sel_rep["Hours"].values
        od = sel_rep["OD"].values
        ax.scatter(hours, od, s=30, alpha=0.5, color=palette[idx])
        keep_mask = compute_mask(hours, od)
        fit_hours = hours[keep_mask]
        row_rep = params[params["ID"] == rep].iloc[0]
        L_r, r_r, t0_r, L0_r = row_rep["L"], row_rep["r"], row_rep["t0"], row_rep["L0"]
        t_dense_rep = np.linspace(fit_hours.min(), fit_hours.max(), 300)
        y_dense_rep = lgf(L_r, r_r, t0_r, L0_r, t_dense_rep)
        ax.plot(t_dense_rep, y_dense_rep, color=palette[idx], linewidth=2, alpha=0.9, label=rep)
    ax.set_title(f"Fitted curves\n{sample_id}", fontsize=16)
    ax.set_xlabel("Time (hours)", size=16)
    ax.set_xlim(0, 22)
    ax.set_ylim(0, 1.5)

    for ax in axes:
        ax.tick_params(axis='both', labelsize=16)

    plt.tight_layout()
    plt.savefig(f'../../out/figures/growth_curves_{sample_id}.png', dpi=150,
                bbox_inches='tight', transparent=True)
    plt.savefig(f'../../out/figures/growth_curves_{sample_id}.svg', dpi=150,
                bbox_inches='tight', transparent=True)
    plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because